In [21]:
import os
import time

import chromadb
import numpy as np
import pandas as pd
from cohere import ClientV2
from dotenv import load_dotenv


In [2]:
load_dotenv()
cohere_api_key = os.getenv('COHERE_API_KEY')
if not cohere_api_key:
    raise ValueError("COHERE_API_KEY not found in .env file")

co = ClientV2(api_key=cohere_api_key)

In [3]:
df = pd.read_csv('arxiv_papers_5k.csv')
embeddings = np.load('embeddings_cohere_5k.npy')

print(f"Loaded {len(df)} papers")
print(f"Embeddings shape: {embeddings.shape}")
print(f"\nPapers per category:")
print(df['category'].value_counts().sort_index())

### Check what metadata we have
print(f"\nAvailable metadata columns:")
print(df.columns.tolist())

Loaded 5000 papers
Embeddings shape: (5000, 1536)

Papers per category:
category
cs.CL    1000
cs.CV    1000
cs.DB    1000
cs.LG    1000
cs.SE    1000
Name: count, dtype: int64

Available metadata columns:
['title', 'abstract', 'authors', 'published', 'category', 'arxiv_id']


In [ ]:
"""
Designing effective metadata schemas
"""
# Good metadata design makes search powerful and performant. it is;

"""
- Filterable: Choose values that match how users actually search.
  If users filter by publication year, store year as an integer.
  If they filter by topic, store normalized category strings.

- Atomic: Store individual fields separately rather than dumping
  everything into a single JSON blob. Want to filter by year?
  Don't make ChromaDB parse "Published: 2024-03-15" from a text field.

- Indexed: Most vector databases index metadata fields differently
  than vector embeddings. Keep metadata fields small and specific so indexing
  works efficiently.

- Consistent: Use the same data types and formats across all documents.
  Don't store year as "2024" for one paper and "March 2024" for another.
"""

"""
Avoid storing:

- Long text in metadata fields: The paper abstract is content, not metadata.
  Store it as the document text, not in a metadata field.

- Nested structures: ChromaDB supports nested metadata,
  but complex JSON trees are hard to filter and often signal confused schema design.

- Redundant information: If you can derive a field from another
  (like "decade" from "year"), consider computing it at query time instead of storing it.

- ?? Frequently changing values: Metadata updates can be expensive.
  Don't store view counts or frequently updated statistics in metadata.
"""

'\n- Filterable: Choose values that match how users actually search.\n  If users filter by publication year, store year as an integer.\n  If they filter by topic, store normalized category strings.\n\n- Atomic: Store individual fields separately rather than dumping\n  everything into a single JSON blob. Want to filter by year?\n  Don\'t make ChromaDB parse "Published: 2024-03-15" from a text field.\n\n- Indexed: Most vector databases index metadata fields differently\n  than vector embeddings. Keep metadata fields small and specific so indexing\n  works efficiently.\n\n- Consistent: Use the same data types and formats across all documents.\n  Don\'t store year as "2024" for one paper and "March 2024" for another.\n'

In [7]:
def prepare_metadata(df):
    """
    Prepare metadata for ChromaDB from our dataframe.

    Returns list of metadata dictionaries, one per paper.
    """
    metadatae = []

    for _, row in df.iterrows():
        # Extract year from published date (format: YYYY-MM-DD)
        year = int(str(row['published'])[:4])

        # Truncate authors if too long (ChromaDB has reasonable limits)
        authors = row['authors'][:200] if len(row['authors']) <= 200 else row['authors'][:197] + "..."

        metadata = {
            'title': row['title'],
            'category': row['category'],
            'year': year,  # Store as integer for range queries
            'authors': authors
        }
        metadatae.append(metadata)

    return metadatae



In [8]:
### Prepare metadata for all papers
metadatae = prepare_metadata(df)

### Check a sample
print("Sample metadata:")
print(metadatae[0])

Sample metadata:
{'title': 'Optimizing Mixture of Block Attention', 'category': 'cs.LG', 'year': 2025, 'authors': 'Guangxuan Xiao, Junxian Guo, Kasra Mazaheri, Song Han'}


In [10]:
"""
a common source of frustration:
if a document is missing a metadata field, ChromaDB's filters won't match it at all.
If filtered by {"year": {"$gte": 2024}} and some papers lack a year field,
those papers simply won't appear in results.
"""
### Initialize ChromaDB client
client = chromadb.Client()

In [11]:
client = chromadb.PersistentClient(path="./chroma_20260331_db")

In [12]:
### Delete existing collection if present (useful for experimentation)
try:
    client.delete_collection(name="arxiv_with_metadata")
    print("Deleted existing collection")
except:
    pass  # Collection didn't exist, that's fine

### Create collection with metadata
collection = client.create_collection(
    name="arxiv_with_metadata",
    metadata={
        "description": "5000 arXiv papers with rich metadata for filtering",
        "hnsw:space": "cosine"  # Using cosine similarity
    }
)

print(f"Created collection: {collection.name}")

Created collection: arxiv_with_metadata


In [13]:
### Prepare data for insertion
ids = [f"paper_{i}" for i in range(len(df))]
documents = df['abstract'].tolist()

### Insert with metadata
### Our 5000 papers fit in one batch (limit is ~5,461)
print(f"Inserting {len(df)} papers with metadata...")

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatae
)

print(f"✓ Collection contains {collection.count()} papers with metadata")

Inserting 5000 papers with metadata...
✓ Collection contains 5000 papers with metadata


In [14]:
### First, let's create a helper function for queries
def search_with_filter(query_text, where_clause=None, n_results=5):
    """
    Search with optional metadata filtering.

    Args:
        query_text: The search query
        where_clause: Optional ChromaDB where clause for filtering
        n_results: Number of results to return

    Returns:
        Search results
    """
    # Embed the query
    response = co.embed(
        texts=[query_text],
        model='embed-v4.0',
        input_type='search_query',
        embedding_types=['float']
    )
    query_embedding = np.array(response.embeddings.float_[0])

    # Search with optional filter
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results,
        where=where_clause  # Apply metadata filter here
    )

    return results


In [15]:

### Example: Search for "deep learning optimization" only in ML papers
query = "deep learning optimization techniques"

results_filtered = search_with_filter(
    query,
    where_clause={"category": "cs.LG"}  # Only Machine Learning papers
)

print(f"Query: '{query}'")
print("Filter: category = 'cs.LG'")
print("\nTop 5 results:")
for i in range(len(results_filtered['ids'][0])):
    metadata = results_filtered['metadatas'][0][i]
    distance = results_filtered['distances'][0][i]

    print(f"\n{i+1}. {metadata['title']}")
    print(f"   Category: {metadata['category']} | Year: {metadata['year']}")
    print(f"   Distance: {distance:.4f}")

Query: 'deep learning optimization techniques'
Filter: category = 'cs.LG'

Top 5 results:

1. Adam symmetry theorem: characterization of the convergence of the stochastic Adam optimizer
   Category: cs.LG | Year: 2025
   Distance: 0.6435

2. Non-Euclidean SGD for Structured Optimization: Unified Analysis and Improved Rates
   Category: cs.LG | Year: 2025
   Distance: 0.6556

3. Training Neural Networks at Any Scale
   Category: cs.LG | Year: 2025
   Distance: 0.6660

4. Deep Progressive Training: scaling up depth capacity of zero/one-layer models
   Category: cs.LG | Year: 2025
   Distance: 0.6669

5. DP-AdamW: Investigating Decoupled Weight Decay and Bias Correction in Private Deep Learning
   Category: cs.LG | Year: 2025
   Distance: 0.6736


In [ ]:
### Search for papers from 2024 or later
results_recent = search_with_filter(
    "neural network architectures",
    where_clause={"year": {"$gte": 2024}}  # Greater than or equal to 2024
)


Recent papers (2024+) about neural network architectures:
1. Bearing Syntactic Fruit with Stack-Augmented Neural Networks (2025)
2. Preparation of Fractal-Inspired Computational Architectures for Advanced Large Language Model Analysis (2025)
3. Preparation of Fractal-Inspired Computational Architectures for Advanced Large Language Model Analysis (2025)


In [17]:

print("Recent papers (2024+) about neural network architectures:")
for i in range(3):  # Show top 3
    metadata = results_recent['metadatas'][0][i]
    print(f"{i+1}. {metadata['title']} ({metadata['year']}) ({metadata['category']})")

Recent papers (2024+) about neural network architectures:
1. Bearing Syntactic Fruit with Stack-Augmented Neural Networks (2025) (cs.CL)
2. Preparation of Fractal-Inspired Computational Architectures for Advanced Large Language Model Analysis (2025) (cs.LG)
3. Preparation of Fractal-Inspired Computational Architectures for Advanced Large Language Model Analysis (2025) (cs.CV)


In [18]:
# power from combining muliple filers
### Find Computer Vision papers from 2025
results_combined = search_with_filter(
    "image recognition and classification",
    where_clause={
        "$and": [
            {"category": "cs.CV"},
            {"year": {"$eq": 2025}}
        ]
    }
)

In [19]:
print("Computer Vision papers from 2025 about image recognition:")
for i in range(3):
    metadata = results_combined['metadatas'][0][i]
    print(f"{i+1}. {metadata['title']}")
    print(f"   {metadata['category']} | {metadata['year']}")

Computer Vision papers from 2025 about image recognition:
1. SWAN -- Enabling Fast and Mobile Histopathology Image Annotation through Swipeable Interfaces
   cs.CV | 2025
2. Covariance Descriptors Meet General Vision Encoders: Riemannian Deep Learning for Medical Image Classification
   cs.CV | 2025
3. UniADC: A Unified Framework for Anomaly Detection and Classification
   cs.CV | 2025


In [20]:
### Papers from either Database or Software Engineering categories
where_db_or_se = {
    "$or": [
        {"category": "cs.DB"},
        {"category": "cs.SE"}
    ]
}

In [22]:
def benchmark_filter(where_clause, n_iterations=100, description=""):
    """
    Benchmark query performance with a specific filter.

    Args:
        where_clause: The filter to apply (None for unfiltered)
        n_iterations: Number of times to run the query
        description: Description of what we're testing

    Returns:
        Average query time in milliseconds
    """
    # Use a fixed query embedding to keep comparisons fair
    query_text = "machine learning model training"
    response = co.embed(
        texts=[query_text],
        model='embed-v4.0',
        input_type='search_query',
        embedding_types=['float']
    )
    query_embedding = np.array(response.embeddings.float_[0])

    # Warm up (run once to load any caches)
    collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5,
        where=where_clause
    )

    # Benchmark
    start_time = time.time()
    for _ in range(n_iterations):
        collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=5,
            where=where_clause
        )
    elapsed = time.time() - start_time
    avg_ms = (elapsed / n_iterations) * 1000

    print(f"{description}")
    print(f"  Average query time: {avg_ms:.2f} ms")
    return avg_ms

In [23]:
print("Running filtering performance benchmarks (100 iterations each)...")
print("=" * 70)

### Baseline: No filtering
baseline_ms = benchmark_filter(None, description="Baseline (no filter)")

Running filtering performance benchmarks (100 iterations each)...
Baseline (no filter)
  Average query time: 3.71 ms


In [24]:
### Category filter
category_ms = benchmark_filter(
    {"category": "cs.LG"},
    description="Category filter (category = 'cs.LG')"
)
category_overhead = (category_ms / baseline_ms)
print(f"  Overhead: {category_overhead:.1f}x slower ({(category_overhead-1)*100:.0f}%)")

Category filter (category = 'cs.LG')
  Average query time: 8.47 ms
  Overhead: 2.3x slower (128%)


In [25]:
### Year range filter
year_ms = benchmark_filter(
    {"year": {"$gte": 2024}},
    description="Year range filter (year >= 2024)"
)
year_overhead = (year_ms / baseline_ms)
print(f"  Overhead: {year_overhead:.1f}x slower ({(year_overhead-1)*100:.0f}%)")

Year range filter (year >= 2024)
  Average query time: 16.09 ms
  Overhead: 4.3x slower (334%)


In [26]:
### Combined filter
combined_ms = benchmark_filter(
    {"$and": [{"category": "cs.LG"}, {"year": {"$gte": 2024}}]},
    description="Combined filter (category AND year)"
)
combined_overhead = (combined_ms / baseline_ms)
print(f"  Overhead: {combined_overhead:.1f}x slower ({(combined_overhead-1)*100:.0f}%)")

Combined filter (category AND year)
  Average query time: 13.01 ms
  Overhead: 3.5x slower (251%)


In [ ]:
"""
What these numbers tell us:

- Unfiltered queries are fast: Our baseline of 4.45ms means ChromaDB's HNSW index works well.

- Category filtering costs 3.3x overhead: The query still completes in 14.82ms,
  which is totally usable, but it's noticeably slower than unfiltered search.

- Numeric range queries are most expensive: Year filtering at 8x overhead
  (35.67ms) shows that range queries on numeric fields are particularly costly in ChromaDB.

- Combined filters fall in between: At 5x overhead (22.34ms),
  combining filters doesn't just multiply the costs. There's some optimization happening.

- Real-world variability: If you run these benchmarks yourself,
  you'll see the exact numbers vary between runs.
  We saw category filtering range from 13.8-16.1ms across multiple benchmark sessions.
  This variability is normal.
  ******** What stays consistent is the order: year filters ********
  ******** are always most expensive, then combined filters, then category filters. ********
"""

In [27]:
from rank_bm25 import BM25Okapi
import string

def simple_tokenize(text):
    """
    Basic tokenization for BM25.

    Lowercase text, remove punctuation, split on whitespace.
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

### Tokenize all abstracts
print("Building BM25 index from 5000 abstracts...")
tokenized_corpus = [simple_tokenize(abstract) for abstract in df['abstract']]

### Create BM25 index
bm25 = BM25Okapi(tokenized_corpus)
print("✓ BM25 index created")

### Test it with a sample query
query = "SQL query optimization indexing"
tokenized_query = simple_tokenize(query)

### Get BM25 scores for all documents
bm25_scores = bm25.get_scores(tokenized_query)

### Find top 5 papers by BM25 score
top_indices = np.argsort(bm25_scores)[::-1][:5]

print(f"\nQuery: '{query}'")
print("Top 5 by BM25 keyword matching:")
for rank, idx in enumerate(top_indices, 1):
    score = bm25_scores[idx]
    title = df.iloc[idx]['title']
    category = df.iloc[idx]['category']
    print(f"{rank}. [{category}] {title[:60]}...")
    print(f"   BM25 Score: {score:.2f}")

Building BM25 index from 5000 abstracts...
✓ BM25 index created

Query: 'SQL query optimization indexing'
Top 5 by BM25 keyword matching:
1. [cs.DB] Learned Adaptive Indexing...
   BM25 Score: 13.34
2. [cs.DB] LLM4Hint: Leveraging Large Language Models for Hint Recommen...
   BM25 Score: 13.25
3. [cs.DB] Cortex AISQL: A Production SQL Engine for Unstructured Data...
   BM25 Score: 12.83
4. [cs.LG] Cortex AISQL: A Production SQL Engine for Unstructured Data...
   BM25 Score: 12.83
5. [cs.DB] A Functional Data Model and Query Language is All You Need...
   BM25 Score: 11.91


In [ ]:
"""
The rank-bm25 library works great for our 5,000 abstracts and similar small datasets.
It's perfect for learning BM25 concepts without complexity.
For larger datasets or production systems, you'd typically use faster BM25
implementations found in search engines like Elasticsearch, OpenSearch,
or Apache Lucene. These are optimized for millions of documents and high query volumes.
For now, rank-bm25 gives us everything we need to understand how keyword
search complements vector search.
"""

In [28]:
### Vector search for the same query
results_vector = search_with_filter(query, n_results=5)

print(f"\nSame query: '{query}'")
print("Top 5 by vector similarity:")
for i in range(5):
    metadata = results_vector['metadatas'][0][i]
    distance = results_vector['distances'][0][i]
    print(f"{i+1}. [{metadata['category']}] {metadata['title'][:60]}...")
    print(f"   Distance: {distance:.4f}")


Same query: 'SQL query optimization indexing'
Top 5 by vector similarity:
1. [cs.DB] VIDEX: A Disaggregated and Extensible Virtual Index for the ...
   Distance: 0.5505
2. [cs.DB] AMAZe: A Multi-Agent Zero-shot Index Advisor for Relational ...
   Distance: 0.5581
3. [cs.DB] AutoIndexer: A Reinforcement Learning-Enhanced Index Advisor...
   Distance: 0.5594
4. [cs.DB] LLM4Hint: Leveraging Large Language Models for Hint Recommen...
   Distance: 0.5852
5. [cs.DB] Training-Free Query Optimization via LLM-Based Plan Similari...
   Distance: 0.5868


In [29]:
# NOTE: Implementing weighted hybrid search

def hybrid_search(query_text, alpha=0.5, n_results=10):
    """
    Combine BM25 keyword search with vector similarity search.

    Args:
        query_text: The search query
        alpha: Weight for BM25 (0 = pure vector, 1 = pure keyword)
        n_results: Number of results to return

    Returns:
        Combined results with hybrid scores
    """
    # Get BM25 scores
    tokenized_query = simple_tokenize(query_text)
    bm25_scores = bm25.get_scores(tokenized_query)

    # Get vector similarities (we'll search more to ensure good coverage)
    response = co.embed(
        texts=[query_text],
        model='embed-v4.0',
        input_type='search_query',
        embedding_types=['float']
    )
    query_embedding = np.array(response.embeddings.float_[0])

    vector_results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=100  # Get more candidates for better coverage
    )

    # Extract vector distances and convert to similarities
    # ChromaDB returns cosine distance (0 to 2, lower = more similar)
    # We'll convert to similarity scores where higher = better for easier combination
    vector_distances = {}
    for i, paper_id in enumerate(vector_results['ids'][0]):
        distance = vector_results['distances'][0][i]
        # Convert distance to similarity (simple inversion)
        similarity = 1 / (1 + distance)
        vector_distances[paper_id] = similarity

    # Normalize BM25 scores to 0-1 range
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    min_bm25 = min(bm25_scores)
    bm25_normalized = {}
    for i, score in enumerate(bm25_scores):
        paper_id = f"paper_{i}"
        normalized = (score - min_bm25) / (max_bm25 - min_bm25) if max_bm25 > min_bm25 else 0
        bm25_normalized[paper_id] = normalized

    # Combine scores using weighted average
    # hybrid_score = alpha * bm25 + (1 - alpha) * vector
    hybrid_scores = {}
    all_paper_ids = set(bm25_normalized.keys()) | set(vector_distances.keys())

    for paper_id in all_paper_ids:
        bm25_score = bm25_normalized.get(paper_id, 0)
        vector_score = vector_distances.get(paper_id, 0)

        hybrid_score = alpha * bm25_score + (1 - alpha) * vector_score
        hybrid_scores[paper_id] = hybrid_score

    # Get top N by hybrid score
    top_paper_ids = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n_results]

    # Format results
    results = []
    for paper_id, score in top_paper_ids:
        paper_idx = int(paper_id.split('_')[1])
        results.append({
            'paper_id': paper_id,
            'title': df.iloc[paper_idx]['title'],
            'category': df.iloc[paper_idx]['category'],
            'abstract': df.iloc[paper_idx]['abstract'][:200] + "...",
            'hybrid_score': score,
            'bm25_score': bm25_normalized.get(paper_id, 0),
            'vector_score': vector_distances.get(paper_id, 0)
        })

    return results


In [30]:

### Test with different alpha values
query = "neural network training optimization"

print(f"Query: '{query}'")
print("=" * 80)

### Pure vector (alpha = 0)
print("\nPure Vector Search (alpha=0.0):")
results = hybrid_search(query, alpha=0.0, n_results=5)
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['category']}] {r['title'][:60]}...")
    print(f"   Hybrid: {r['hybrid_score']:.3f} (Vector: {r['vector_score']:.3f}, BM25: {r['bm25_score']:.3f})")

### Hybrid 30% keyword, 70% vector
print("\nHybrid 30/70 (alpha=0.3):")
results = hybrid_search(query, alpha=0.3, n_results=5)
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['category']}] {r['title'][:60]}...")
    print(f"   Hybrid: {r['hybrid_score']:.3f} (Vector: {r['vector_score']:.3f}, BM25: {r['bm25_score']:.3f})")

### Hybrid 50/50
print("\nHybrid 50/50 (alpha=0.5):")
results = hybrid_search(query, alpha=0.5, n_results=5)
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['category']}] {r['title'][:60]}...")
    print(f"   Hybrid: {r['hybrid_score']:.3f} (Vector: {r['vector_score']:.3f}, BM25: {r['bm25_score']:.3f})")

### Pure keyword (alpha = 1.0)
print("\nPure BM25 Keyword (alpha=1.0):")
results = hybrid_search(query, alpha=1.0, n_results=5)
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['category']}] {r['title'][:60]}...")
    print(f"   Hybrid: {r['hybrid_score']:.3f} (Vector: {r['vector_score']:.3f}, BM25: {r['bm25_score']:.3f})")

Query: 'neural network training optimization'

Pure Vector Search (alpha=0.0):
1. [cs.LG] Training Neural Networks at Any Scale...
   Hybrid: 0.642 (Vector: 0.642, BM25: 0.749)
2. [cs.LG] On the Convergence of Overparameterized Problems: Inherent P...
   Hybrid: 0.630 (Vector: 0.630, BM25: 1.000)
3. [cs.LG] A Distributed Training Architecture For Combinatorial Optimi...
   Hybrid: 0.618 (Vector: 0.618, BM25: 0.884)
4. [cs.LG] Adam symmetry theorem: characterization of the convergence o...
   Hybrid: 0.617 (Vector: 0.617, BM25: 0.381)
5. [cs.LG] Can Training Dynamics of Scale-Invariant Neural Networks Be ...
   Hybrid: 0.609 (Vector: 0.609, BM25: 0.566)

Hybrid 30/70 (alpha=0.3):
1. [cs.LG] On the Convergence of Overparameterized Problems: Inherent P...
   Hybrid: 0.741 (Vector: 0.630, BM25: 1.000)
2. [cs.LG] Neuronal Fluctuations: Learning Rates vs Participating Neuro...
   Hybrid: 0.713 (Vector: 0.603, BM25: 0.971)
3. [cs.CV] NeuCLIP: Efficient Large-Scale CLIP Training with Neural No

In [ ]:
# Evaluating search strategies systematically
"""
Category precision @k: What percentage of the top k results are in the expected category?
"""

In [32]:
test_queries = [
    {
        "text": "natural language processing transformers",
        "expected_category": "cs.CL",
        "description": "NLP query"
    },
    {
        "text": "image segmentation computer vision",
        "expected_category": "cs.CV",
        "description": "Vision query"
    },
    {
        "text": "database query optimization indexing",
        "expected_category": "cs.DB",
        "description": "Database query"
    },
    {
        "text": "neural network training deep learning",
        "expected_category": "cs.LG",
        "description": "ML query with clear terms"
    },
    {
        "text": "software testing debugging quality assurance",
        "expected_category": "cs.SE",
        "description": "Software engineering query"
    },
    {
        "text": "attention mechanisms sequence models",
        "expected_category": "cs.CL",
        "description": "NLP architecture query"
    },
    {
        "text": "convolutional neural networks image recognition",
        "expected_category": "cs.CV",
        "description": "Vision with technical terms"
    },
    {
        "text": "distributed systems database consistency",
        "expected_category": "cs.DB",
        "description": "Database systems query"
    },
    {
        "text": "reinforcement learning policy gradient",
        "expected_category": "cs.LG",
        "description": "RL query"
    },
    {
        "text": "code review static analysis",
        "expected_category": "cs.SE",
        "description": "SE development query"
    }
]

print(f"Created {len(test_queries)} test queries")
print("Expected category distribution:")
categories = [q['expected_category'] for q in test_queries]
print(pd.Series(categories).value_counts().sort_index())

Created 10 test queries
Expected category distribution:
cs.CL    2
cs.CV    2
cs.DB    2
cs.LG    2
cs.SE    2
Name: count, dtype: int64


In [33]:
def calculate_category_precision(query_text, expected_category, search_type="vector", alpha=0.5):
    """
    Calculate what percentage of top 5 results match expected category.

    Args:
        query_text: The search query
        expected_category: Expected category (e.g., 'cs.LG')
        search_type: 'vector', 'bm25', or 'hybrid'
        alpha: Weight for BM25 if using hybrid

    Returns:
        Precision (0.0 to 1.0)
    """
    if search_type == "vector":
        results = search_with_filter(query_text, n_results=5)
        categories = [r['category'] for r in results['metadatas'][0]]

    elif search_type == "bm25":
        tokenized_query = simple_tokenize(query_text)
        bm25_scores = bm25.get_scores(tokenized_query)
        top_indices = np.argsort(bm25_scores)[::-1][:5]
        categories = [df.iloc[idx]['category'] for idx in top_indices]

    elif search_type == "hybrid":
        results = hybrid_search(query_text, alpha=alpha, n_results=5)
        categories = [r['category'] for r in results]

    # Calculate precision
    matches = sum(1 for cat in categories if cat == expected_category)
    precision = matches / len(categories)

    return precision, categories


In [34]:

### Evaluate all strategies
results_summary = {
    'Pure Vector': [],
    'Hybrid 30/70': [],
    'Hybrid 50/50': [],
    'Pure BM25': []
}

print("Evaluating search strategies on 10 test queries...")
print("=" * 80)

for query_info in test_queries:
    query = query_info['text']
    expected = query_info['expected_category']

    print(f"\nQuery: {query}")
    print(f"Expected: {expected}")

    # Pure vector
    precision, _ = calculate_category_precision(query, expected, "vector")
    results_summary['Pure Vector'].append(precision)
    print(f"  Pure Vector: {precision*100:.0f}% precision")

    # Hybrid 30/70
    precision, _ = calculate_category_precision(query, expected, "hybrid", alpha=0.3)
    results_summary['Hybrid 30/70'].append(precision)
    print(f"  Hybrid 30/70: {precision*100:.0f}% precision")

    # Hybrid 50/50
    precision, _ = calculate_category_precision(query, expected, "hybrid", alpha=0.5)
    results_summary['Hybrid 50/50'].append(precision)
    print(f"  Hybrid 50/50: {precision*100:.0f}% precision")

    # Pure BM25
    precision, _ = calculate_category_precision(query, expected, "bm25")
    results_summary['Pure BM25'].append(precision)
    print(f"  Pure BM25: {precision*100:.0f}% precision")

### Calculate average precision for each strategy
print("\n" + "=" * 80)
print("OVERALL RESULTS")
print("=" * 80)
for strategy, precisions in results_summary.items():
    avg_precision = sum(precisions) / len(precisions)
    print(f"{strategy}: {avg_precision*100:.0f}% average category precision")

Evaluating search strategies on 10 test queries...

Query: natural language processing transformers
Expected: cs.CL
  Pure Vector: 80% precision
  Hybrid 30/70: 60% precision
  Hybrid 50/50: 60% precision
  Pure BM25: 60% precision

Query: image segmentation computer vision
Expected: cs.CV
  Pure Vector: 80% precision
  Hybrid 30/70: 100% precision
  Hybrid 50/50: 100% precision
  Pure BM25: 80% precision

Query: database query optimization indexing
Expected: cs.DB
  Pure Vector: 100% precision
  Hybrid 30/70: 100% precision
  Hybrid 50/50: 100% precision
  Pure BM25: 80% precision

Query: neural network training deep learning
Expected: cs.LG
  Pure Vector: 60% precision
  Hybrid 30/70: 40% precision
  Hybrid 50/50: 60% precision
  Pure BM25: 80% precision

Query: software testing debugging quality assurance
Expected: cs.SE
  Pure Vector: 100% precision
  Hybrid 30/70: 100% precision
  Hybrid 50/50: 100% precision
  Pure BM25: 100% precision

Query: attention mechanisms sequence models

In [ ]:
"""
✔️ Pure Vector Performed Best on This Dataset
✔️ Adding BM25 keyword matching introduced false positives
"""

In [ ]:
"""
When hybrid might outperform pure vector:

- Searching structured data (product catalogs, API documentation)
- Queries with rare technical terms that might not embed well
- Domains where exact keyword presence is meaningful
- Documents with inconsistent writing quality where semantic meaning is unclear
"""


In [ ]:
"""
Some queries fail across ALL strategies.
For example, testing "reducing storage requirements for system event data" hoping
to find a paper about log compression. None of the approaches found it. Why?

The query used "reducing storage requirements" but the paper said
"compression" and "resource savings."
These are semantically equivalent, but the vocabulary differs.
At 5k scale with multiple papers legitimately matching each query,
vocabulary mismatches become visible.

This isn't a failure of vector search or hybrid search.
It's the reality of semantic retrieval: users search with general terms,
papers use technical jargon. Sometimes the gap is too wide.
"""

In [ ]:
"""
Throughout the evaluation, it's noticed that well-crafted queries with
clear technical terms perform well across all strategies,
while vague queries struggle everywhere.

A query like "neural network training optimization techniques" succeeds
because it used the same language papers use.
A query like "making models work better" fails because it's too general
and uses informal language.

The lesson: Before optimizing the search strategy,
make sure the queries match how the documents are written.
Understanding the corpus matters more than choosing between vector and keyword search.
"""

In [ ]:
"""
Decision frameworks for filtering and hybrid search

When to Use Metadata Filtering
    Use filtering when:
        - Users explicitly request filters ("show me papers from 2024")
        - Filtering meaningfully improves result quality
        - Your query volume is manageable (ChromaDB can handle dozens of filtered queries per second)
        - The performance cost is acceptable for your use case
    
    Design your schema carefully:
        - Store filterable fields as atomic values (integers for years, strings for categories)
        - Avoid nested JSON blobs or long text in metadata
        - Keep metadata consistent across documents
        - Test filtering performance on your actual data before deploying
    
    Accept the overhead:
        - Filtered queries run slower than unfiltered ones in ChromaDB
        - This is a characteristic of how ChromaDB approaches the problem
        - Production databases handle filtering with different tradeoffs
        - Design for the database actually being used

When to Consider Hybrid Search
    Try hybrid search when:
        - Your documents have structured fields where exact matches matter
        - Queries include rare technical terms that might not embed well
        - Testing shows hybrid outperforms pure vector on your test queries
        - You can afford the implementation and maintenance complexity
    
    Stick with pure vector when:
        - Your documents have rich natural language (like our academic abstracts)
        - Vector search already achieves high precision on test queries
        - Simplicity and maintainability matter
        - Your embedding model captures domain terminology well
    
    The decision framework:

        1. Build pure vector search first

        2. Create representative test queries

        3. Measure precision/recall on pure vector

        4. Only if results are inadequate, implement hybrid

        5. Compare hybrid against pure vector on same test queries
        
        6. Choose the approach with measurably better results

!!!Don't add complexity without evidence it helps.!!!
"""

In [35]:
my_queries = [
    "your domain-specific query here",
    "another query relevant to your work",
    # Add more
]

for query in my_queries:
    print(f"\n{'='*70}")
    print(f"Query: {query}")

    # Try pure vector
    results_vector = search_with_filter(query, n_results=5)

    # Try hybrid
    results_hybrid = hybrid_search(query, alpha=0.5, n_results=5)

    # Compare the categories returned
    # Which approach surfaces more relevant papers?


Query: your domain-specific query here

Query: another query relevant to your work


In [ ]:
query = "your challenging query here"

for alpha in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    results = hybrid_search(query, alpha=alpha, n_results=5)
    categories = [r['category'] for r in results]

    print(f"Alpha={alpha}: {categories}")
    # Which alpha gives the best results for this query?

In [ ]:
### Try various filter patterns
filters_to_test = [
    {"category": "cs.LG"},
    {"year": {"$gte": 2024}},
    {"category": "cs.LG", "year": {"$eq": 2025}},
    {"$or": [{"category": "cs.LG"}, {"category": "cs.CV"}]}
]

query = "deep learning applications"

for where_clause in filters_to_test:
    results = search_with_filter(query, where_clause, n_results=5)
    categories = [r['category'] for r in results['metadatas'][0]]
    print(f"Filter {where_clause}: {categories}")

In [ ]:
### If you have expertise in a specific field,
### create queries where you KNOW which papers should match

domain_specific_queries = [
    {
        "text": "your expert query",
        "expected_category": "cs.XX",
        "notes": "why this query should return this category"
    },
    # Add more
]

### Run evaluation and see which strategy performs best
### on YOUR domain-specific queries

In [ ]:
"""
Key Takeaways:

- Metadata schema design matters: store filterable fields as atomic,
  consistent values and ensure all documents have the same fields

- Filtering adds overhead in ChromaDB (category cheapest, year range most expensive,
  combined in between)

- Pure vector achieved 84% category precision vs 78% for hybrid/BM25 on
  academic abstracts due to rich vocabulary

- Hybrid search has value in specific scenarios (structured data, rare keywords)
  but adds complexity.

- Vocabulary mismatch between queries and documents affects all search strategies equally

- Start with pure vector search, measure systematically, add complexity only when data justifies it

- ChromaDB teaches filtering concepts; production databases optimize differently

- Evaluation frameworks with test queries matter more than assumptions about "best practices"
"""